# Train PhytoDex flower classifier
This notebook trains a MobileNetV2 classification head on five flower categories. It does not train the typed assistant, identify all plants, or diagnose disease. Run cells in order. Select Runtime → Change runtime type → T4 GPU if available. CPU also works, more slowly.


## 1 Upload the supplied kit ZIP
Choose `phytodex-ai-kit.zip`. Code is run in subprocesses so package changes do not require importing TensorFlow into this notebook process.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()
kit_name = next(name for name in uploaded if name.endswith('.zip'))
with zipfile.ZipFile(kit_name) as z:
    base = Path('/content/phytodex-work').resolve()
    for name in z.namelist():
        if not (base / name).resolve().is_relative_to(base):
            raise ValueError('Unsafe ZIP path')
    z.extractall(base)
os.chdir(base / 'phytodex-ai-kit')
print(Path.cwd())


## 2 Install training dependencies
No API key is needed. Do not install these heavy training packages on the Pi.

In [ ]:
import subprocess, sys
def run(*args):
    subprocess.run([sys.executable, *args], check=True)
run('-m', 'pip', 'install', '-r', 'ml/requirements-train.txt')


## 3 Download and split the dataset
Downloads about 218 MiB from TensorFlow. Keeps source/license information, removes exact decoded-image duplicates, and creates fixed train, validation and test groups. Folder names become the model labels. This dataset does not guarantee that different photos show different specimens.

In [ ]:
run('ml/prepare_data.py')


## 4 Train and export
ImageNet supplies the starting visual features. We train the classification head, evaluate held-out images, export float16 weights with float32 image input, and verify that exported predictions match. This cell can take several minutes or longer depending on the assigned machine.

In [ ]:
run('ml/train.py', '--epochs', '8')


## 5 Inspect real metrics
`test.accuracy` is held-out dataset accuracy, not real-camera performance. `threshold=1.01` means validation did not support candidate acceptance: the service will always say uncertain. Do not lower it just to make the demo look confident.

In [ ]:
import json
print(json.dumps(json.loads(Path('models/flowers/metrics.json').read_text()), indent=2))
print(Path('models/flowers/metadata.json').read_text())


## 6 Download the complete results
Keep this ZIP outside Git. It contains the model, labels, metadata, metrics, split manifest and available dataset attribution. The deployment guide copies only the small runtime bundle to the Pi.

In [ ]:
import shutil
shutil.make_archive('/content/phytodex-training-results', 'zip', 'models')
files.download('/content/phytodex-training-results.zip')
